# Complete BaBar isobar CP closure with Square Dalitz normalization

This notebook implements the complete nominal BaBar signal model of Phys. Rev. D 78, 012004 (2008), arXiv:0803.4451. The Cartesian CP convention is
\[c_j^+=(x_j+\Delta x_j)+i(y_j+\Delta y_j),\qquad c_j^-=(x_j-\Delta x_j)+i(y_j-\Delta y_j).\]

All floating Cartesian coefficient parameters are **unbounded**. The fit uses the joint charge + Dalitz likelihood with denominator $I_+ + I_-$. The Square-Dalitz normalization grid is explicitly uniform in $(m',\theta')\in[0,1]^2$. The single randomized start is generated around the injected truth to avoid the asymptotic coefficient-scale runaway seen with broad unbounded starts.

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    BaBarFlatte, CPRealImag, DecayChannel, DecayModel, LASS, Minimizer,
    NonResonant, Parameter, Resonance, SquareDalitzGrid, enable_x64,
    invariants_to_square_dalitz, weighted_resample,
)
from dalitzplotfitter.likelihood import CPJointNLL

enable_x64()

## 1. BaBar Table-I Cartesian CP coefficients

In [ ]:
TABLE_I = {
    # name: (x, y, dx, dy, fixed_xy, fixed_cp)
    'Kstar892':    ( 1.000,  0.000, -0.017, -0.238, True,  False),
    'KpiS':        ( 1.718, -0.727, -0.154, -0.285, False, False),
    'rho770':      ( 0.683, -0.025, -0.160,  0.169, False, False),
    'f0_980':      (-0.220,  1.203, -0.109,  0.047, False, False),
    'chic0':       (-0.263,  0.180, -0.033, -0.007, False, False),
    'NR':          (-0.594,  0.068,  0.000,  0.000, False, True),
    'K2star1430':  (-0.301,  0.424,  0.032,  0.007, False, False),
    'omega782':    (-0.058,  0.100,  0.000,  0.000, False, True),
    'f2_1270':     (-0.193,  0.110, -0.089,  0.125, False, False),
    'fX1300':      (-0.290, -0.136,  0.024,  0.056, False, False),
}

def cp_coefficient(name):
    x, y, dx, dy, fixed_xy, fixed_cp = TABLE_I[name]
    def p(suffix, value, fixed=False, step=0.02):
        return Parameter.coefficient(
            f'{name}.{suffix}', value, owner=name, fixed=fixed,
            bounds=None, step=step,
        )
    return CPRealImag(
        p('x', x, fixed_xy), p('y', y, fixed_xy),
        p('dx', dx, fixed_cp, 0.01), p('dy', dy, fixed_cp, 0.01),
    )

coeff = {name: cp_coefficient(name) for name in TABLE_I}
for c in coeff.values():
    for p in c.parameters:
        if not p.fixed:
            assert p.bounds is None

## 2. Complete nominal dynamical model

In [ ]:
channel_plus  = DecayChannel('B+', ('K+', 'pi+', 'pi-'))
channel_minus = DecayChannel('B-', ('K-', 'pi-', 'pi+'))

def build_model(channel, charge):
    c = lambda name: coeff[name].for_charge(charge)
    R = 4.0
    return DecayModel(channel, [
        Resonance('Kstar892',   (0,2), c('Kstar892'),   mass=0.8958, width=0.0474, spin=1, resonance_radius=R, parent_radius=R),
        Resonance('KpiS',       (0,2), c('KpiS'),       lineshape=LASS(2.07,3.32,1.8), mass=1.425, width=0.270, spin=0, resonance_radius=R, parent_radius=R),
        Resonance('rho770',     (1,2), c('rho770'),     mass=0.7753, width=0.1491, spin=1, resonance_radius=R, parent_radius=R),
        Resonance('f0_980',     (1,2), c('f0_980'),     lineshape=BaBarFlatte(), mass=0.965, width=0.0, spin=0, resonance_radius=R, parent_radius=R),
        Resonance('chic0',      (1,2), c('chic0'),      mass=3.4147, width=0.0105, spin=0, resonance_radius=R, parent_radius=R),
        NonResonant(c('NR'), name='NR'),
        Resonance('K2star1430', (0,2), c('K2star1430'), mass=1.4324, width=0.109, spin=2, resonance_radius=R, parent_radius=R),
        Resonance('omega782',   (1,2), c('omega782'),   mass=0.78265, width=0.00849, spin=1, resonance_radius=R, parent_radius=R),
        Resonance('f2_1270',    (1,2), c('f2_1270'),    mass=1.2755, width=0.1867, spin=2, resonance_radius=R, parent_radius=R),
        Resonance('fX1300',     (1,2), c('fX1300'),     mass=1.479, width=0.080, spin=0, resonance_radius=R, parent_radius=R),
    ], normalization_resolution=350)

model_plus = build_model(channel_plus, +1)
model_minus = build_model(channel_minus, -1)
truth = {p.name: float(p.value) for p in model_plus.parameters}
parameters = model_plus.parameters
print('free parameters:', sum(not p.fixed for p in parameters))
print('all free parameters unbounded:', all(p.bounds is None for p in parameters if not p.fixed))

## 3. Uniform Square-Dalitz normalization grid

In [ ]:
SDP_RESOLUTION = 450
SDP_PAIR = (0, 2)

square_norm_plus = SquareDalitzGrid(
    channel_plus.parent_mass, channel_plus.daughter_masses,
    resolution=SDP_RESOLUTION, pair=SDP_PAIR, quadrature='midpoint',
).sample()
square_norm_minus = SquareDalitzGrid(
    channel_minus.parent_mass, channel_minus.daughter_masses,
    resolution=SDP_RESOLUTION, pair=SDP_PAIR, quadrature='midpoint',
).sample()

mp_grid, tp_grid = invariants_to_square_dalitz(
    square_norm_plus.s12, square_norm_plus.s13, square_norm_plus.s23,
    mother_mass=channel_plus.parent_mass, masses=channel_plus.daughter_masses, pair=SDP_PAIR,
)
print('normalization points:', square_norm_plus.size)
print('mprime range:', float(mp_grid.min()), float(mp_grid.max()))
print('thetaprime range:', float(tp_grid.min()), float(tp_grid.max()))
print('Jacobian min/max:', float(square_norm_plus.weights.min()), float(square_norm_plus.weights.max()))

## 4. Generate one joint charge + Dalitz toy

In [ ]:
N_POOL, N_TOTAL = 500_000, 120_000
pool_plus = model_plus.generate_phase_space(N_POOL, seed=78012004)
pool_minus = model_minus.generate_phase_space(N_POOL, seed=78012005)
proposal_cache_plus = model_plus.prepare_cache(pool_plus, normalization_sample=square_norm_plus)
proposal_cache_minus = model_minus.prepare_cache(pool_minus, normalization_sample=square_norm_minus)

w_plus = pool_plus.weights * proposal_cache_plus.intensity(truth)
w_minus = pool_minus.weights * proposal_cache_minus.intensity(truth)
I_plus = float(proposal_cache_plus.normalization(truth))
I_minus = float(proposal_cache_minus.normalization(truth))
p_plus = I_plus / (I_plus + I_minus)

rng = np.random.default_rng(78012006)
N_PLUS = rng.binomial(N_TOTAL, p_plus)
N_MINUS = N_TOTAL - N_PLUS
print(f'I+={I_plus:.6f}, I-={I_minus:.6f}, P(+)={p_plus:.5f}')
print(f'N+={N_PLUS}, N-={N_MINUS}, raw asym={(N_MINUS-N_PLUS)/N_TOTAL:+.5f}')

toy_plus = weighted_resample(jax.random.key(78012007), pool_plus, w_plus, N_PLUS, replace=True)
toy_minus = weighted_resample(jax.random.key(78012008), pool_minus, w_minus, N_MINUS, replace=True)

## 5. Charge-separated Dalitz plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), constrained_layout=True)
for ax, toy, title in [(axes[0], toy_plus, r'$B^+$'), (axes[1], toy_minus, r'$B^-$')]:
    h = ax.hist2d(np.asarray(toy.s13), np.asarray(toy.s23), bins=110)
    fig.colorbar(h[3], ax=ax, label='events')
    ax.set(xlabel=r'$s_{13}$ [GeV$^2$]', ylabel=r'$s_{23}$ [GeV$^2$]', title=title)
plt.show()

## 6. Square-Dalitz plots

In [ ]:
def square_coordinates(sample, channel):
    return invariants_to_square_dalitz(
        sample.s12, sample.s13, sample.s23,
        mother_mass=channel.parent_mass, masses=channel.daughter_masses, pair=SDP_PAIR,
    )

mp_plus, tp_plus = square_coordinates(toy_plus, channel_plus)
mp_minus, tp_minus = square_coordinates(toy_minus, channel_minus)
fig, axes = plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
for ax, mp, tp, title in [(axes[0],mp_plus,tp_plus,r'$B^+$ SqDP'),(axes[1],mp_minus,tp_minus,r'$B^-$ SqDP')]:
    h=ax.hist2d(np.asarray(mp),np.asarray(tp),bins=100,range=((0,1),(0,1)))
    fig.colorbar(h[3],ax=ax,label='events')
    ax.set(xlabel=r'$m^\prime$',ylabel=r'$\theta^\prime$',title=title,xlim=(0,1),ylim=(0,1))
plt.show()

## 7. Joint CP likelihood and stabilized randomized start

In [ ]:
cache_plus = model_plus.prepare_cache(toy_plus, normalization_sample=square_norm_plus)
cache_minus = model_minus.prepare_cache(toy_minus, normalization_sample=square_norm_minus)
objective = CPJointNLL(cache_plus, cache_minus)

fitter = Minimizer(objective, parameters, tolerance=0.1, verbose=1)

START_SEED = 20260830
START_SIGMA_XY = 0.30
START_SIGMA_CP = 0.08
rng = np.random.default_rng(START_SEED)
start = {}
for p in parameters:
    if p.fixed:
        continue
    sigma = START_SIGMA_CP if p.name.endswith(('.dx', '.dy')) else START_SIGMA_XY
    start[p.name] = float(truth[p.name] + rng.normal(0.0, sigma))

print('NLL(truth) =', float(objective(truth)))
print('NLL(start) =', float(objective(start)))
print('start sigmas: xy=', START_SIGMA_XY, ' cp=', START_SIGMA_CP)
gradient_check = fitter.check_gradient(start, step_scale=1e-5, print_table=True)

## 8. Initial distributions after randomization

The comparison includes both ordinary Dalitz variables ($s_{13}$ and $s_{23}$) and the Square-Dalitz variables ($m'$ and $\theta'$). Each panel shows toy, injected truth and randomized start before minimization.

In [ ]:
def projection_values(sample, channel, variable):
    if variable in ('s13', 's23'):
        return np.asarray(getattr(sample, variable))
    mp, tp = square_coordinates(sample, channel)
    if variable == 'mprime':
        return np.asarray(mp)
    if variable == 'thetaprime':
        return np.asarray(tp)
    raise ValueError(variable)

def weighted_projection(pool, cache, channel, values, variable, bins):
    intensity = np.asarray(cache.intensity(values))
    weights = np.asarray(pool.weights) * intensity
    coordinates = projection_values(pool, channel, variable)
    hist, _ = np.histogram(coordinates, bins=bins, weights=weights)
    return hist

def comparison_histograms(toy, pool, cache, channel, variable, model_values):
    data_values = projection_values(toy, channel, variable)
    pool_values = projection_values(pool, channel, variable)
    if variable in ('mprime', 'thetaprime'):
        bins = np.linspace(0.0, 1.0, 81)
    else:
        bins = np.linspace(min(data_values.min(), pool_values.min()), max(data_values.max(), pool_values.max()), 90)
    centers = 0.5*(bins[:-1] + bins[1:])
    hdata, _ = np.histogram(data_values, bins=bins)
    model_hists = []
    for values in model_values:
        h = weighted_projection(pool, cache, channel, values, variable, bins)
        h *= hdata.sum()/h.sum()
        model_hists.append(h)
    return centers, hdata, model_hists

variables = [
    ('s13', r'$s_{13}$ [GeV$^2$]'),
    ('s23', r'$s_{23}$ [GeV$^2$]'),
    ('mprime', r'$m^\prime$'),
    ('thetaprime', r'$\theta^\prime$'),
]

for toy, pool, cache, channel, charge_title in [
    (toy_plus, pool_plus, proposal_cache_plus, channel_plus, r'$B^+$'),
    (toy_minus, pool_minus, proposal_cache_minus, channel_minus, r'$B^-$'),
]:
    fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
    for ax, (variable, xlabel) in zip(axes.flat, variables):
        centers, hdata, (htruth, hstart) = comparison_histograms(
            toy, pool, cache, channel, variable, [truth, start]
        )
        ax.errorbar(centers, hdata, yerr=np.sqrt(np.maximum(hdata,1)), fmt='.', label='toy')
        ax.step(centers, htruth, where='mid', ls='--', label='truth')
        ax.step(centers, hstart, where='mid', label='randomized start')
        ax.set(xlabel=xlabel, ylabel='events', title=f'{charge_title}: {xlabel}')
        ax.legend()
    plt.show()

## 9. Perform exactly one fit and reject runaway solutions

In [ ]:
result = fitter.fit(start_values=start, simplex=False, ncall=50000)
free = [p for p in parameters if not p.fixed]
fit = {p.name: float(result.values[p.name]) for p in free}

nll_truth = float(objective(truth))
nll_start = float(objective(start))
nll_fit = float(result.fval)
max_abs_fit = max(abs(v) for v in fit.values())
runaway = (not np.isfinite(nll_fit)) or max_abs_fit > 100.0 or nll_fit > nll_start

print('valid          =', bool(result.valid))
print('NLL(start)     =', nll_start)
print('NLL(truth)     =', nll_truth)
print('NLL(fit)       =', nll_fit)
print('fit-truth NLL  =', nll_fit - nll_truth)
print('EDM            =', float(result.fmin.edm))
print('function calls =', int(result.nfcn))
print('max |fit par|  =', max_abs_fit)
print('runaway        =', runaway)
print('charge probabilities truth:', tuple(float(v) for v in objective.charge_probabilities(truth)))

if runaway:
    raise RuntimeError('CP fit entered a runaway/non-physical coefficient-scale direction.')

## 10. Pulls of all floating Cartesian parameters

In [ ]:
rows=[]
print(f"{'parameter':18s} {'truth':>10s} {'start':>10s} {'fit':>10s} {'error':>10s} {'pull':>9s}")
for p in free:
    t=float(truth[p.name]); s=float(start[p.name]); f=float(result.values[p.name]); e=float(result.errors[p.name])
    pull=(f-t)/e
    rows.append((p.name,t,s,f,e,pull))
    print(f"{p.name:18s} {t:10.5f} {s:10.5f} {f:10.5f} {e:10.5f} {pull:9.3f}")

names=[r[0] for r in rows]
pulls=np.asarray([r[5] for r in rows])
print('max |pull| =', float(np.max(np.abs(pulls))))
print('RMS pull   =', float(np.sqrt(np.mean(pulls**2))))
fig,ax=plt.subplots(figsize=(12,5))
ax.axhline(0); ax.axhline(1,ls='--'); ax.axhline(-1,ls='--')
ax.scatter(np.arange(len(names)),pulls)
ax.set_xticks(np.arange(len(names))); ax.set_xticklabels(names,rotation=70,ha='right'); ax.set_ylabel('pull')
plt.tight_layout(); plt.show()

## 11. Final projections: truth, randomized start and fit

The final comparison is shown in both the ordinary Dalitz variables and the Square-Dalitz variables.

In [ ]:
for toy, pool, cache, channel, charge_title in [
    (toy_plus, pool_plus, proposal_cache_plus, channel_plus, r'$B^+$'),
    (toy_minus, pool_minus, proposal_cache_minus, channel_minus, r'$B^-$'),
]:
    fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
    for ax, (variable, xlabel) in zip(axes.flat, variables):
        centers, hdata, (htruth, hstart, hfit) = comparison_histograms(
            toy, pool, cache, channel, variable, [truth, start, fit]
        )
        ax.errorbar(centers, hdata, yerr=np.sqrt(np.maximum(hdata,1)), fmt='.', label='toy')
        ax.step(centers, htruth, where='mid', ls='--', label='truth')
        ax.step(centers, hstart, where='mid', label='randomized start')
        ax.step(centers, hfit, where='mid', label='fit')
        ax.set(xlabel=xlabel, ylabel='events', title=f'{charge_title}: {xlabel}')
        ax.legend()
    plt.show()

## 12. Argand truth versus start versus fit

In [ ]:
fig, ax = plt.subplots(figsize=(7,7))
for name in TABLE_I:
    x_t,y_t,dx_t,dy_t,_,_=TABLE_I[name]
    def get_from(values, suffix, default): return float(values.get(f'{name}.{suffix}', default))
    x_s=get_from(start,'x',x_t); y_s=get_from(start,'y',y_t); dx_s=get_from(start,'dx',dx_t); dy_s=get_from(start,'dy',dy_t)
    x_f=get_from(fit,'x',x_t); y_f=get_from(fit,'y',y_t); dx_f=get_from(fit,'dx',dx_t); dy_f=get_from(fit,'dy',dy_t)
    ax.scatter([x_t+dx_t,x_t-dx_t],[y_t+dy_t,y_t-dy_t],marker='x')
    ax.scatter([x_s+dx_s,x_s-dx_s],[y_s+dy_s,y_s-dy_s],marker='s',facecolors='none')
    ax.scatter([x_f+dx_f,x_f-dx_f],[y_f+dy_f,y_f-dy_f],marker='o',facecolors='none')
ax.axhline(0,lw=.5); ax.axvline(0,lw=.5)
ax.set_xlabel('Re(c)'); ax.set_ylabel('Im(c)'); ax.set_title('Truth (x), randomized start (squares), fit (circles)')
plt.show()

## 13. Global charge fractions

In [ ]:
p_truth = tuple(float(v) for v in objective.charge_probabilities(truth))
p_start = tuple(float(v) for v in objective.charge_probabilities(start))
p_fit = tuple(float(v) for v in objective.charge_probabilities(fit))
print('truth P(+), P(-):', p_truth)
print('start P(+), P(-):', p_start)
print('fit   P(+), P(-):', p_fit)
print('observed fractions:', N_PLUS/N_TOTAL, N_MINUS/N_TOTAL)

## Interpretation

A successful closure requires the one fit to remain away from the asymptotic coefficient-scale direction, improve the randomized start, return finite uncertainties and recover both the CP-even $(x,y)$ and CP-odd $(\Delta x,\Delta y)$ parameters with statistically reasonable pulls. The ordinary Dalitz and Square-Dalitz projections make the effect of the randomization and the recovery by the fit visible for both charges.